# Mint-line raster pilot

This notebook evaluates `line_raster_mint_01`. It reconstructs the serpentine trajectory, applies the independently estimated 3.0-second physical response lag, and maps the 32-sensor RMS fractional change from the latter half of flag 1.

The mapped value is **sensor change**, not mint probability. This is a usable pilot, not a definitive reconstruction: there is no geometrically matched blank raster, the source endpoints were not recorded in desk coordinates, and height varied more than the controlled-protocol target.

In [1]:
from pathlib import Path
import csv
import json
import math
import statistics

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

repo_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next((path for path in repo_candidates if (path / 'record_cyranose_reading_pose.py').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from inside the repository.')

TRIAL_ID = 'line_raster_mint_01'
RESPONSE_LAG_S = 3.0
SOURCE_X_CM = -26.7  # independent estimate from accepted lag Pairs 2 and 3
SCAN_DEVICE_START_S = 42.5  # used only to select the hand-reviewed raster interval
SCAN_DEVICE_END_S = 161.0
HEIGHT_BAND_CM = (1.0, 3.5)
MAX_INTERPOLATION_GAP_S = 1.5

def find_session(trial_id):
    matches = []
    for metadata_path in REPO_ROOT.rglob('session_metadata.json'):
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        if metadata.get('trial_id') == trial_id:
            matches.append(metadata_path.parent)
    if len(matches) != 1:
        raise RuntimeError(f'Expected one session for {trial_id!r}; found {len(matches)}.')
    return matches[0]

SESSION_DIR = find_session(TRIAL_ID)
SESSION_DIR

WindowsPath('C:/Users/dell-xps/Documents/realsense-apriltag/experiments/spatial-mapping/line-raster/pilot-usable/cyranose_reading_pose_session_20260722_150515')

In [2]:
with (SESSION_DIR/'cyranose_reading_pose.csv').open(newline='', encoding='utf-8') as source:
    raw_rows = list(csv.DictReader(source))
alignment = json.loads((SESSION_DIR/'alignment_summary.json').read_text(encoding='utf-8'))
first_device_time = float(raw_rows[0]['pcnose_device_time_s'])

flag1 = [row for row in raw_rows if row['pcnose_flag'] == '1']
late_flag1 = flag1[len(flag1)//2:]
sensor_fields = [f'pcnose_S{i}_kohm' for i in range(1, 33)]
baseline = {field: statistics.median(float(row[field]) for row in late_flag1) for field in sensor_fields}

def number(value):
    try:
        parsed = float(value)
    except (TypeError, ValueError):
        return None
    return parsed if math.isfinite(parsed) else None

rows = []
for raw in raw_rows:
    fractional = [(float(raw[field])-baseline[field])/baseline[field]*100 for field in sensor_fields]
    rows.append({
        'flag': raw['pcnose_flag'],
        'device_s': float(raw['pcnose_device_time_s'])-first_device_time,
        'time_s': number(raw.get('pose_elapsed_s')),
        'x': number(raw.get('snout_desk_x_cm')),
        'y': number(raw.get('snout_desk_y_cm')),
        'z': number(raw.get('snout_desk_z_cm')),
        'response': math.sqrt(statistics.mean(value*value for value in fractional)),
    })

pose_rows = [row for row in rows if None not in (row['time_s'], row['x'], row['y'], row['z'])]

def interpolate_pose(target_time, max_gap_s=MAX_INTERPOLATION_GAP_S):
    if target_time < pose_rows[0]['time_s'] or target_time > pose_rows[-1]['time_s']:
        return None
    for before, after in zip(pose_rows, pose_rows[1:]):
        if before['time_s'] <= target_time <= after['time_s']:
            gap = after['time_s']-before['time_s']
            if gap > max_gap_s:
                return None
            fraction = (target_time-before['time_s'])/max(gap, 1e-9)
            return {axis: before[axis]+fraction*(after[axis]-before[axis]) for axis in ('x','y','z')}
    return None

scan_rows = [row for row in rows if row['flag'] == '2' and SCAN_DEVICE_START_S <= row['device_s'] <= SCAN_DEVICE_END_S]
raw_position_rows = [row for row in scan_rows if None not in (row['x'], row['y'], row['z']) and abs(row['z']) < 5]

corrected = []
for row in scan_rows:
    if row['time_s'] is None:
        continue
    pose = interpolate_pose(row['time_s']-RESPONSE_LAG_S)
    if pose is None or not (HEIGHT_BAND_CM[0] <= abs(pose['z']) <= HEIGHT_BAND_CM[1]):
        continue
    if -43 <= pose['x'] <= -10 and -56 <= pose['y'] <= -27:
        corrected.append({**pose, 'height': abs(pose['z']), 'response': row['response'], 'time_s': row['time_s']})

print(f"Digital matches: {alignment['matched_readings']}/{alignment['total_readings']} within 250 ms")
print(f"Alignment p95: {alignment['absolute_pose_minus_pcnose_ms']['p95']:.1f} ms")
print(f'Raster readings with desk position: {len(raw_position_rows)}/{len(scan_rows)}')
print(f'Retained after 3 s lag, height, and interpolation-gap QC: {len(corrected)}')

Digital matches: 235/235 within 250 ms
Alignment p95: 91.2 ms
Raster readings with desk position: 123/133
Retained after 3 s lag, height, and interpolation-gap QC: 117


In [3]:
def smoothed_grid(samples, step=1.0, radius=3.0, sigma=1.5):
    grid_x = [value for value in range(-43, -9)]
    grid_y = [value for value in range(-56, -26)]
    grid_z = []
    for y in grid_y:
        row_values = []
        for x in grid_x:
            weighted = total = 0.0
            neighbors = 0
            for sample in samples:
                distance2 = (sample['x']-x)**2 + (sample['y']-y)**2
                if distance2 <= radius*radius:
                    weight = math.exp(-0.5*distance2/(sigma*sigma))
                    weighted += weight*sample['response']
                    total += weight
                    neighbors += 1
            row_values.append(weighted/total if neighbors >= 2 and total else None)
        grid_z.append(row_values)
    return grid_x, grid_y, grid_z

grid_x, grid_y, grid_z = smoothed_grid(corrected)
figure = make_subplots(rows=1, cols=2, subplot_titles=('Tracked raster', '3.0 s lag-corrected sensor response'))
figure.add_trace(go.Scatter(
    x=[row['x'] for row in raw_position_rows], y=[row['y'] for row in raw_position_rows],
    mode='lines+markers', marker=dict(size=4), line=dict(width=1.5),
    hovertemplate='x %{x:.2f} cm<br>y %{y:.2f} cm<extra></extra>', showlegend=False,
), row=1, col=1)
figure.add_trace(go.Heatmap(
    x=grid_x, y=grid_y, z=grid_z, colorscale='Cividis', connectgaps=False,
    colorbar=dict(title='RMS change (%)'),
    hovertemplate='x %{x:.1f} cm<br>y %{y:.1f} cm<br>response %{z:.3f}%<extra></extra>',
), row=1, col=2)
figure.add_trace(go.Scatter(
    x=[row['x'] for row in corrected], y=[row['y'] for row in corrected],
    mode='lines', line=dict(color='rgba(230,230,230,.55)', width=1), hoverinfo='skip', showlegend=False,
), row=1, col=2)
figure.add_vline(x=SOURCE_X_CM, line_dash='dash', line_width=1.5, row=1, col=1)
figure.add_vline(x=SOURCE_X_CM, line_dash='dash', line_width=1.5, row=1, col=2)
figure.update_xaxes(title_text='Desk X (cm)', row=1, col=1)
figure.update_xaxes(title_text='Desk X (cm)', row=1, col=2)
figure.update_yaxes(title_text='Desk Y (cm)', scaleanchor='x', scaleratio=1, row=1, col=1)
figure.update_yaxes(title_text='Desk Y (cm)', scaleanchor='x2', scaleratio=1, row=1, col=2)
figure.update_layout(height=590, width=1120, margin=dict(l=60, r=80, t=70, b=60))
figure.show()

In [4]:
def spatial_contrast(lag_s):
    near, far = [], []
    for row in scan_rows:
        if row['time_s'] is None:
            continue
        pose = interpolate_pose(row['time_s']-lag_s)
        if pose is None or not (HEIGHT_BAND_CM[0] <= abs(pose['z']) <= HEIGHT_BAND_CM[1]):
            continue
        distance = abs(pose['x']-SOURCE_X_CM)
        if distance < 3:
            near.append(row['response'])
        elif distance >= 8:
            far.append(row['response'])
    if len(near) < 5 or len(far) < 5:
        return None
    return statistics.median(near)/statistics.median(far), statistics.median(near), statistics.median(far)

lag_points = []
for step in range(21):
    lag = step*0.25
    result = spatial_contrast(lag)
    if result:
        lag_points.append((lag, *result))

working = spatial_contrast(RESPONSE_LAG_S)
uncorrected = spatial_contrast(0.0)
best = max(lag_points, key=lambda point: point[1])

contrast_figure = go.Figure(go.Scatter(
    x=[point[0] for point in lag_points], y=[point[1] for point in lag_points],
    mode='lines+markers', marker=dict(size=6),
    hovertemplate='lag %{x:.2f} s<br>near/far %{y:.2f}×<extra></extra>',
))
contrast_figure.add_vline(x=RESPONSE_LAG_S, line_dash='dash', line_width=1.5, annotation_text='working lag')
contrast_figure.update_xaxes(title='Assumed physical lag (s)')
contrast_figure.update_yaxes(title='Median near-source / far-background response')
contrast_figure.update_layout(height=390, width=850, showlegend=False, margin=dict(l=80, r=40, t=40, b=60))
contrast_figure.show()

heights = [row['height'] for row in corrected]
display(Markdown(f"""**Pilot result.** Without correction, near-source response was only **{uncorrected[0]:.2f}×** far background. With the independent 3.0 s correction it became **{working[0]:.2f}×** ({working[1]:.3f}% versus {working[2]:.3f}% median RMS change). The diagnostic maximum occurred near **{best[0]:.2f} s**, which supports—but does not independently prove—the earlier lag estimate.

Height after QC had median **{statistics.median(heights):.2f} cm**. This is promising first spatial evidence, but a matched blank raster and recorded source endpoints are still required before claiming accurate line recovery."""))

**Pilot result.** Without correction, near-source response was only **0.60×** far background. With the independent 3.0 s correction it became **2.01×** (0.591% versus 0.295% median RMS change). The diagnostic maximum occurred near **2.75 s**, which supports—but does not independently prove—the earlier lag estimate.

Height after QC had median **2.19 cm**. This is promising first spatial evidence, but a matched blank raster and recorded source endpoints are still required before claiming accurate line recovery.